# Popularidade x qualidade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import ttest_ind, mannwhitneyu

sns.set_theme(style="whitegrid")

# Seed para reprodutibilidade dos testes de permutação
RNG = np.random.default_rng(42)


In [ ]:
movies = pd.read_parquet('db/movies.parquet')

ratings = pd.read_parquet('db/ratings.parquet')


movies = movies.merge(
    ratings,
    on='tconst',
    how='inner'
)

movies = movies.dropna(
   subset=['primaryTitle', 'startYear', 'averageRating', 'numVotes']
)
# Combinar tabelas separadas para poder realizar a análise
movies.head()

In [ ]:
patterns = [
    r"(?:\s|:)(2|3|4|5|6|7|8|9)\b",
    r"\bpart\b",
    r"\bchapter\b",
    r"\breturns\b",
    r"\bii\b",
    r"\biii\b",
    r"\biv\b",
    r"\bv\b",
    r"\bvi\b",
    r"\bvs\b",
    r"\borigins\b",
    r"\blegacy\b",
    r"\breloaded\b",
    r"\brevolutions\b",
    r"\bfinal\b"
]

regex = re.compile("|".join(patterns), re.IGNORECASE)

movies['franquia'] = movies['primaryTitle'].apply(
    lambda x: bool(regex.search(str(x)))
)

# Rótulo legível para usar nos gráficos (eixos com True/False são confusos)
movies['grupo'] = movies['franquia'].map({
    False: 'Original',
    True: 'Franquia'
})


In [ ]:
originais = movies[movies['franquia'] == False].copy()
franquias = movies[movies['franquia'] == True].copy()

print(f"Filmes originais : {len(originais)}")
print(f"Filmes franquia  : {len(franquias)}")


# Popularidade
Calculada por `numVotes`


In [ ]:
# Problema: Como fazer para visualizar se podem ter filmes com 10 votos e filmes com 1000000 votos?
# Solução: Usar uma transformação nos dados para melhorar a visualização. No caso Log10 para "normalizar as grandezas".
movies['logVotes'] = np.log10(movies['numVotes'] + 1)
originais['logVotes'] = np.log10(originais['numVotes'] + 1)
franquias['logVotes'] = np.log10(franquias['numVotes'] + 1)


In [ ]:
plt.figure(figsize=(10,6))

sns.kdeplot(originais['logVotes'],
            label='Originais',
            fill=True)

sns.kdeplot(franquias['logVotes'],
            label='Franquias',
            fill=True)

plt.xlabel('Número de votos - log10')
plt.ylabel('Porcentagem da amostra')
plt.title('Distribuição da Popularidade')
plt.legend()
plt.show()


# Qualidade

Calculada por `averageRating`.


In [ ]:
plt.figure(figsize=(10,6))

sns.kdeplot(originais['averageRating'],
            label='Originais',
            fill=True)

sns.kdeplot(franquias['averageRating'],
            label='Franquias',
            fill=True)

plt.xlabel('averageRating')
plt.ylabel('Densidade')
plt.title('Distribuição das Notas')
plt.legend()
plt.show()


# Teste A/B: Originais x Franquias

Comparamos as distribuições de **qualidade** (`averageRating`) e **popularidade** (`logVotes`) entre os dois grupos usando três abordagens:

- **Teste t de Welch**: referência paramétrica, assume normalidade (provavelmente violada aqui).
- **Mann-Whitney U**: teste não-paramétrico, principal critério de decisão do projeto.
- **Teste de permutação**: embaralha os rótulos de grupo N vezes e recalcula a diferença de médias, gerando um p-valor empírico sem assumir distribuição teórica.

H0 em todos os casos: as duas distribuições (Original x Franquia) têm a mesma média/posição central.


In [ ]:
def permutation_test_diff_means(a, b, n_perm=10_000, rng=RNG):
    """
    Teste de permutação para diferença de médias entre dois grupos.
    H0: não há diferença entre os grupos (rótulo é irrelevante).
    Retorna a diferença observada e o p-valor bicaudal.
    """
    a = np.asarray(a)
    b = np.asarray(b)

    observed_diff = a.mean() - b.mean()

    pooled = np.concatenate([a, b])
    n_a = len(a)

    perm_diffs = np.empty(n_perm)

    for i in range(n_perm):
        shuffled = rng.permutation(pooled)
        perm_diffs[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()

    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

    return observed_diff, p_value, perm_diffs


In [ ]:
print("="*60)
print("TESTE A/B - FILMES ORIGINAIS x FRANQUIAS")
print("="*60)

# ==========================
# QUALIDADE (averageRating)
# ==========================

t_rating, p_rating = ttest_ind(
    originais['averageRating'],
    franquias['averageRating'],
    equal_var=False
)

u_rating, p_mw_rating = mannwhitneyu(
    originais['averageRating'],
    franquias['averageRating'],
    alternative='two-sided'
)

diff_rating, p_perm_rating, _ = permutation_test_diff_means(
    originais['averageRating'],
    franquias['averageRating']
)

print("\nQUALIDADE")
print(f"Média Originais : {originais['averageRating'].mean():.2f}")
print(f"Média Franquias : {franquias['averageRating'].mean():.2f}")
print(f"Diferença (Originais - Franquias) : {diff_rating:.3f}")
print(f"Teste t (Welch)     : p = {p_rating:.5f}")
print(f"Mann-Whitney        : p = {p_mw_rating:.5f}")
print(f"Permutação (10.000) : p = {p_perm_rating:.5f}")

if p_mw_rating < 0.05:
    print("Conclusão: Existe diferença estatisticamente significativa na qualidade.")
else:
    print("Conclusão: Não foi encontrada diferença significativa na qualidade.")

# ==========================
# POPULARIDADE (logVotes)
# ==========================

t_votes, p_votes = ttest_ind(
    originais['logVotes'],
    franquias['logVotes'],
    equal_var=False
)

u_votes, p_mw_votes = mannwhitneyu(
    originais['logVotes'],
    franquias['logVotes'],
    alternative='two-sided'
)

diff_votes, p_perm_votes, _ = permutation_test_diff_means(
    originais['logVotes'],
    franquias['logVotes']
)

print("\nPOPULARIDADE")
print(f"Média Originais : {originais['logVotes'].mean():.2f}")
print(f"Média Franquias : {franquias['logVotes'].mean():.2f}")
print(f"Diferença (Originais - Franquias) : {diff_votes:.3f}")
print(f"Teste t (Welch)     : p = {p_votes:.5f}")
print(f"Mann-Whitney        : p = {p_mw_votes:.5f}")
print(f"Permutação (10.000) : p = {p_perm_votes:.5f}")

if p_mw_votes < 0.05:
    print("Conclusão: Existe diferença estatisticamente significativa na popularidade.")
else:
    print("Conclusão: Não foi encontrada diferença significativa na popularidade.")

# ==========================
# BOXPLOTS
# ==========================

plt.figure(figsize=(7,5))
sns.boxplot(data=movies, x='grupo', y='averageRating')
plt.title('Qualidade: Filmes Originais x Franquias')
plt.xlabel('')
plt.ylabel('Nota média')
plt.show()

plt.figure(figsize=(7,5))
sns.boxplot(data=movies, x='grupo', y='logVotes')
plt.title('Popularidade: Filmes Originais x Franquias')
plt.xlabel('')
plt.ylabel('Log10(Número de votos)')
plt.show()
